In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "mps"
print(torch.backends.mps.is_available())

True


In [2]:
ds = load_dataset("cais/mmlu", "high_school_physics", split="test")
row = ds[0]
print(row["question"])
print(row["choices"])
print(row["answer"])

The plates of a capacitor are charged to a potential difference of 5 V. If the capacitance is 2 mF, what is the charge on the positive plate?
['0.005 C', '0.01 C', '0.02 C', '0.5 C']
1


In [ ]:
model_name = "allenai/OLMo-2-0425-1B"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16).to(device)
model.eval()

W1125 22:41:32.958000 41883 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/179 [00:00<?, ?it/s]

Olmo2ForCausalLM(
  (model): Olmo2Model(
    (embed_tokens): Embedding(100352, 2048, padding_idx=100277)
    (layers): ModuleList(
      (0-15): 16 x Olmo2DecoderLayer(
        (self_attn): Olmo2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Olmo2RMSNorm((2048,), eps=1e-06)
          (k_norm): Olmo2RMSNorm((2048,), eps=1e-06)
        )
        (mlp): Olmo2MLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (post_attention_layernorm): Olmo2RMSNorm((2048,), eps=1e-06

In [4]:
letters = ["A", "B", "C", "D"]

def build_prompt(row):
    text = row["question"] + "\n"
    for letter, choice in zip(letters, row["choices"]):
        text += f"{letter}. {choice}\n"
    text += "Answer:"
    return text

prompt = build_prompt(row)
print(prompt)

The plates of a capacitor are charged to a potential difference of 5 V. If the capacitance is 2 mF, what is the charge on the positive plate?
A. 0.005 C
B. 0.01 C
C. 0.02 C
D. 0.5 C
Answer:


In [5]:
letter_ids = []
for letter in letters:
    ids = tok.encode(" " + letter, add_special_tokens=False)
    print(letter, ids)
    letter_ids.append(ids[0])

A [362]
B [426]
C [356]
D [423]


In [6]:
inputs = tok(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])

next_token_scores = out.logits[0, -1]
letter_scores = next_token_scores[letter_ids].float()
probs = torch.softmax(letter_scores, dim=-1)

for letter, p in zip(letters, probs.tolist()):
    print(f"{letter}: {p:.3f}")
print("Sum:", probs.sum().item())
print("Model picks:", letters[probs.argmax().item()], "| Correct:", letters[row["answer"]])

A: 0.171
B: 0.220
C: 0.437
D: 0.171
Sum: 1.0
Model picks: C | Correct: B
